In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModel,AutoTokenizer,AutoModelForSeq2SeqLM
embedding_model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
generator_model_name="google/flan-t5-small"
embed_tokenizer=AutoTokenizer.from_pretrained(embedding_model_name)
embed_model=AutoModel.from_pretrained(embedding_model_name)
gen_tokenizer=AutoTokenizer.from_pretrained(generator_model_name)
gen_model=AutoModelForSeq2SeqLM.from_pretrained(generator_model_name)
documents=[
    "For fat loss dinner,choose high protien,low fat,and moderate carbs",
    "Good chest exercises include bench press,incline dumbbell press,and cable fly",
    "Squats train the quadriceps,glutes,and core stability",
    "To gain high muscle,you need enough protien,calorie surplus,and progressive overload"
]
def encode_texts(texts):
    inputs=embed_tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs=embed_model(**inputs)
        token_embeddings=outputs.last_hidden_state
        attention_mask=inputs["attention_mask"]
        mask=attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        masked_embeddings=token_embeddings*mask

        sum_embeddings=torch.sum(masked_embeddings,dim=1)
        sum_mask=torch.clamp(mask.sum(dim=1),min=1e-9)

        sentence_embeddings=sum_embeddings/sum_mask
        sentence_embeddings=F.normalize(sentence_embeddings,p=2,dim=1)

        return sentence_embeddings
doc_embeddings=encode_texts(documents)
def retrieve_documents(query,documents,doc_embeddings,top_k=2):
    query_embeddings=encode_texts([query])
    scores=torch.matmul(query_embeddings,doc_embeddings.T)
    topk_values,topk_indices=torch.topk(scores,k=top_k,dim=1)
    retrieved_doc=[]
    for i in range(top_k):
        idx=topk_indices[0][i].item()
        score=topk_values[0][i].item()
        retrieved_doc.append({
            "documents":documents[idx],
            "score":score
        })
    return retrieved_doc
def build_prompt(query,retrieved_doc):
    context="\n".join(
        [f"-{item['documents']}" for item in retrieved_doc]
    )
    prompt=(
        "Answer the question using the context.\n"
        "If the context is relevant,use it clearly and briefly.\n\n"
        f"Context:\n{context}\n\n"
        f"Question:{query}\n"
        "Answer:"
    )
    return prompt
def generate_answer(prompt):
    inputs=gen_tokenizer(prompt,return_tensors="pt",truncation=True)
    with torch.no_grad():
        output_ids=gen_model.generate(
            **inputs,
            max_new_tokens=64
        )
    answer=gen_tokenizer.decode(output_ids[0],skip_special_tokens=True)
    return answer
def rag_pipeline(query,documents,doc_embeddings,top_k=2):
    retrieved_doc=retrieve_documents(query,documents,doc_embeddings,top_k=top_k)
    prompt=build_prompt(query,retrieved_doc)
    answer=generate_answer(prompt)
    return{
        "query":query,
        "retrieved_doc":retrieved_doc,
        "prompt":prompt,
        "answer":answer
    }
query="What should I eat for dinner during  fat loss?"
result=rag_pipeline(query,documents,doc_embeddings,top_k=2)
print("Query",result["query"])
print("\nRetrieve doc:")
for i,item in enumerate(result["retrieved_doc"]):
    print(f"Rank{i+1}")
    print("documents",item["documents"])
    print("score",round(item["score"],4))
    print()
print("Prompt:\n")
print(result["prompt"])
print("\nGenereted answer;\n")
print(result["answer"])

SyntaxError: f-string: unmatched '[' (1481400988.py, line 53)

In [ ]:
"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
"google/flan-t5-small"
"For fat loss dinner,choose high protien,low fat,and moderate carbs"
"Good chest exercises include bench press,incline dumbbell press,and cable fly"
"Squats train the quadriceps,glutes,and core stability"
"To gain high muscle,you need enough protien,calorie surplus,and progressive overload"
"Answer the question using the context.\n"
"If the context is relevant,use it clearly and briefly.\n\n"
f"Context:\n{context}\n\n"
f"Question:{query}\n"
"Answer:"
"What should I eat for dinner during  fat loss?"